# Modelo de classificação

Reconstruído em 28/10/2026 a partir da conversa compartilhada e das decisões posteriores do participante. O notebook define o preparo, a SVD e a rede, mas não executa o treinamento automaticamente.

A entrada é a concatenação dos fatores B e C. Existem dez camadas ocultas; cada largura é o piso de dois terços da largura anterior. Todas usam sigmoide. A saída tem um neurônio com sigmoide: alvo 0 para gato e 1 para cachorro. A função de perda é MSE.

Ainda precisam ser definidos k_modelo, taxa de aprendizado, número de épocas, divisão dos dados e tamanho do lote. A interpretação de 0,5 é o ponto médio da saída; esse número, por si só, não garante uma probabilidade calibrada.

In [ ]:
import numpy as np
from PIL import Image
from scipy.sparse.linalg import svds


k_modelo = None  # defina o k da SVD usado pelo modelo
escala_cinza = True  # False: manter RGB
TAMANHO_ALVO = (300, 300)  # largura, altura; recorte quadrado

def preparar_imagem(A, escala_cinza=True):
    """Recorta o centro em um quadrado e redimensiona para 300 x 300 pixels."""
    if not isinstance(escala_cinza, (bool, np.bool_)):
        raise TypeError("escala_cinza deve ser True ou False.")
    if isinstance(A, Image.Image):
        imagem = A
    else:
        matriz = np.asarray(A)
        if matriz.ndim not in (2, 3) or (matriz.ndim == 3 and matriz.shape[2] != 3):
            raise ValueError("A deve ter formato (altura, largura) ou (altura, largura, 3).")
        if matriz.ndim == 3 and matriz.dtype != np.uint8:
            raise TypeError("Uma matriz RGB deve conter pixels uint8.")
        imagem = Image.fromarray(matriz)
    imagem = imagem.convert("L" if escala_cinza else "RGB")
    largura, altura = imagem.size
    lado_recorte = min(largura, altura)
    esquerda = (largura - lado_recorte) // 2
    topo = (altura - lado_recorte) // 2
    imagem = imagem.crop((esquerda, topo, esquerda + lado_recorte, topo + lado_recorte))
    if imagem.size != TAMANHO_ALVO:
        imagem = imagem.resize(TAMANHO_ALVO)
    return np.asarray(imagem, dtype=float)

def fatorar_canal(canal, k):
    canal = np.asarray(canal, dtype=float)
    if canal.ndim != 2:
        raise ValueError("Cada canal deve ser uma matriz bidimensional.")
    if not isinstance(k, (int, np.integer)) or not 1 <= k < min(canal.shape):
        raise ValueError(f"k deve ser inteiro entre 1 e {min(canal.shape) - 1}.")
    U_k, sigma_k, V_k_T = svds(canal, k=k, which="LM")
    ordem_decrescente = np.argsort(sigma_k)[::-1]
    U_k = U_k[:, ordem_decrescente]
    sigma_k = sigma_k[ordem_decrescente]
    V_k_T = V_k_T[ordem_decrescente, :]
    B_k = U_k
    C_k = sigma_k[:, None] * V_k_T
    return B_k, C_k


def transformar(A, k, escala_cinza=True):
    """Retorna (B_k, C_k) em cinza ou seis fatores na ordem RGB."""
    imagem = preparar_imagem(A, escala_cinza=escala_cinza)
    if escala_cinza:
        return fatorar_canal(imagem, k)
    B_R, C_R = fatorar_canal(imagem[:, :, 0], k)
    B_G, C_G = fatorar_canal(imagem[:, :, 1], k)
    B_B, C_B = fatorar_canal(imagem[:, :, 2], k)
    return B_R, C_R, B_G, C_G, B_B, C_B


def montar_vetor_entrada(A):
    """Concatena os fatores definidos por k_modelo em um vetor float32."""
    if not isinstance(k_modelo, int) or not 1 <= k_modelo < 300:
        raise ValueError("Defina k_modelo como inteiro entre 1 e 299.")
    fatores = transformar(A, k_modelo, escala_cinza=escala_cinza)
    return np.concatenate([fator.ravel() for fator in fatores]).astype(np.float32)

## Preparo da imagem e entrada

Cada imagem é recortada no maior quadrado central e redimensionada para 300 × 300 pixels, sem esticar. O redimensionamento usa o padrão do Pillow. O valor de k_modelo e o booleano escala_cinza ficam no início da primeira célula de código.

Em cinza, transformar retorna B_k e C_k, ambos com 300k elementos. montar_vetor_entrada concatena os fatores e produz 600k números. Em RGB, a função retorna B_R, C_R, B_G, C_G, B_B e C_B, produzindo 1800k números após concatenação.

A rede recebe um lote de vetores com formato (quantidade_de_imagens, entrada_dim).

## PyTorch

A conversa incluiu a instalação com `%pip install --user torch torchvision` para um ambiente compartilhado. Execute essa instalação separadamente somente se for necessária no ambiente em que abrir o notebook.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [ ]:
class RedeNeural(nn.Module):
    def __init__(self, entrada_dim):
        super().__init__()
        if not isinstance(entrada_dim, int) or entrada_dim < 1:
            raise ValueError("entrada_dim deve ser um inteiro positivo.")

        self.camadas_ocultas = nn.ModuleList()
        largura_atual = entrada_dim
        for _ in range(10):
            largura_seguinte = (2 * largura_atual) // 3
            if largura_seguinte < 1:
                raise ValueError("A regra de 2/3 produziu uma camada vazia.")
            self.camadas_ocultas.append(nn.Linear(largura_atual, largura_seguinte))
            largura_atual = largura_seguinte

        self.saida = nn.Linear(largura_atual, 1)
        self.sigmoide = nn.Sigmoid()

    def forward(self, x):
        for camada in self.camadas_ocultas:
            x = self.sigmoide(camada(x))
        return self.sigmoide(self.saida(x))

In [ ]:
# Configure k_modelo e escala_cinza na primeira célula de código.
if not isinstance(k_modelo, int) or not 1 <= k_modelo < 300:
    raise ValueError("Defina k_modelo como inteiro entre 1 e 299 na primeira célula.")
entrada_dim = 2 * 300 * k_modelo * (1 if escala_cinza else 3)
modelo = RedeNeural(entrada_dim)

# Defina estes valores antes de chamar treinar.
taxa_aprendizado = None
epocas = None

In [ ]:
def treinar(modelo, dataloader, epocas, taxa_aprendizado):
    criterio = nn.MSELoss()
    otimizador = optim.Adam(modelo.parameters(), lr=taxa_aprendizado)
    historico = []
    for epoca in range(epocas):
        modelo.train()
        perda_total = 0
        total_amostras = 0
        for X, y in dataloader:
            otimizador.zero_grad()
            previsao = modelo(X)
            if y.shape != previsao.shape:
                raise ValueError("y deve ter formato (tamanho_do_lote, 1).")
            y = y.to(dtype=previsao.dtype, device=previsao.device)
            perda = criterio(previsao, y)
            perda.backward()
            otimizador.step()
            quantidade = X.shape[0]
            perda_total += perda.item() * quantidade
            total_amostras += quantidade
        mse_medio = perda_total / total_amostras
        historico.append(mse_medio)
        print(f"Época {epoca + 1}/{epocas} | MSE = {mse_medio:.6f}")
    return historico

In [ ]:
def plotar_treinamento(historico):
    plt.figure(figsize=(10, 5))
    plt.plot(range(1, len(historico) + 1), historico)
    plt.xlabel("Época")
    plt.ylabel("MSE")
    plt.title("Evolução do erro durante o treinamento")
    plt.grid(True)
    plt.tight_layout()
    plt.show()